**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `R/rstudio/R_RUN_ALL_MODELS.Rmd`, `python_prebuilt/`. Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# V06 — Ground validation of the satellite panel (v17.1)

Pairs the benchmark-site ground measurements with the panel at the linked pixels (from P08) and the same `(Year, Season)`:

| ground | satellite | relationship |
|---|---|---|
| LAI (ceptometer) | `LAI`, `NDVI`, `SAVI`, `EVI` | direct (LAI), monotone (VIs) |
| SSM 0–10 cm | `SMDI`, `LSWI`, `NDMI`, `NDWI` | proxy |
| TDR root-zone 0–30 cm | `SMDI`, `LSWI`, `NDMI` | proxy |
| field polygons (cropped) | `LandUse`, `LandUseDW`, Rabi `NDVI`/`LSWI` | cropland class truth (M10 uses `LandUse==2`); Rabi crop presence |

Reports n, Pearson r, Spearman ρ, RMSE, bias and slope by institution and by SWS role (saturation/control). Nothing here changes an estimator.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (v20.55: the engine installs a missing ESTIMATOR package itself -- wheel first, then source -- and confirms 'N of N installed' at every run; these base libraries it needs before it can start)")


In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

# ---- CELL 1: setup ----
import numpy as np, pandas as pd
import _common as C
C.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
C.start_cell_log("V06")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
import _ground_common as G
MODEL_ID = "V06"
RESULTS_DIR = os.path.join(C.RESULTS_ROOT, MODEL_ID); os.makedirs(RESULTS_DIR, exist_ok=True)
STR = {"bm_site_no": str, "borewell_no": str, "probe_or_survey_no": str}
# ground variable -> (csv, value column, links 'variable' name, site-no column, satellite columns to compare)
PLAN = {"lai": ("03_ground_lai_long.csv", "lai_mean", "lai", "bm_site_no", ["LAI", "NDVI", "SAVI", "EVI"]),
        "ssm": ("02_ground_ssm_long.csv", "ssm_mean", "ssm", "bm_site_no", ["SMDI", "LSWI", "NDMI", "NDWI"]),
        "tdr": ("06_ground_tdr_rootzone_0_30cm_by_visit.csv", "moisture_pct", "tdr", "probe_or_survey_no", ["SMDI", "LSWI", "NDMI"])}
C.info("V06 | cell 1: setup done")


In [ ]:
GI = C.ground_inputs_status()
if GI['missing']: C.warn('V06 needs the ground inputs -- stopping'); raise SystemExit
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 2 (v20.46): BM SUB-WATERSHED MEANS vs the satellite mean of each sub-watershed's core, season by season ----
# The mean of a sub-watershed's BM sites represents that sub-watershed (your rule), so the validation is at that scale --
# never site by site. r = correlation over sub-watershed x season rows between the BM mean and the satellite core mean.
import _ground_common as GC
C.step(2, "BM sub-watershed means vs satellite core means")
results = []
def sws_validation():
    fit, val = GC.fit_ground_surrogates(C, write=False, verbose=False, register=False)
    if val is None or not len(val):
        raise C.InsufficientDataError("no sub-watershed x season has both a BM mean and satellite core values "
                                      "(needs the BM ground files, and >= 3 sub-watershed x season rows)")
    val = val.copy(); val.insert(0, "scale", "sub-watershed mean x season")
    results.append(val); C.save_results(val, RESULTS_DIR, "V06_sws_level_validation.csv")
    return val
C.cell_guard(sws_validation)()
C.done(2, next_task="CELL 3 -- cropland-class check over field polygons")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 3: cropland-class check over field-survey polygons (M10 uses LandUse==2) ----
C.step(3, "LandUse class over cropped field polygons")
def landuse_check():
    pl = pd.read_parquet(C.GROUND_LINKS_PATH); pl = pl[pl.link_type == "polygon"]
    if not len(pl): raise C.InsufficientDataError("no polygon links in ground_links.parquet (P08 cell 4)")
    pix = list(pl.pixel_id_nearest.unique())     # native panel dtype
    import pyarrow.parquet as pq, pyarrow.compute as pc, pyarrow as pa
    pf = pq.ParquetFile(C.PREPARED_PANEL); cols = [c for c in ["pixel_id", "Year", "Season", "LandUse", "LandUseDW", "NDVI", "LSWI"] if c in pf.schema_arrow.names]; parts = []
    for i in C.progress(range(pf.num_row_groups), desc="polygon pixels scan"):
        t = pf.read_row_group(i, columns=cols); t = t.filter(pc.is_in(t["pixel_id"], value_set=pa.array(pix)))
        if t.num_rows: parts.append(t.to_pandas())
    pm = pd.concat(parts, ignore_index=True)
    pl = pl.rename(columns={"pixel_id_nearest": "pixel_id"}).dropna(subset=["Year", "Season"])
    for fr in (pl, pm):
        fr["Year"] = fr["Year"].astype(int); fr["Season"] = fr["Season"].astype(int)
    pl["pixel_id"] = pl["pixel_id"].astype(pm["pixel_id"].dtype)
    m = pl.merge(pm, on=["pixel_id", "Year", "Season"], how="inner")
    out = {}
    if "LandUse" in m: out["share_LandUse_eq_2_over_cropped_polygons"] = float((m["LandUse"] == 2).mean())
    if "LandUseDW" in m: out["LandUseDW_class_distribution"] = m["LandUseDW"].value_counts(normalize=True).round(3).to_dict()
    # Rabi crop-presence: NDVI / LSWI in Season 2 vs next_season_cropped
    rb = m[(m.Season == 2)].dropna(subset=["next_season_cropped"]) if "next_season_cropped" in m else pd.DataFrame()
    if len(rb):
        rb["y"] = pd.to_numeric(rb["next_season_cropped"], errors="coerce")
        for v in [c for c in ("NDVI", "LSWI") if c in rb.columns]:
            out[f"rabi_{v}_mean_cropped"] = float(rb.loc[rb.y == 1, v].mean()); out[f"rabi_{v}_mean_not_cropped"] = float(rb.loc[rb.y == 0, v].mean())
    C.save_results(out, RESULTS_DIR, "V06_landuse_cropland_check.csv")
    print(json.dumps(out, indent=1) if False else out)
    return out
lu_check = C.cell_guard(landuse_check)()
C.done(3, next_task="CELL 4 -- summary")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    C = _il.import_module("_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 4: summary ----
C.step(4, "summary")
if results:
    summary = pd.concat(results, ignore_index=True)
    C.save_results(summary, RESULTS_DIR, "V06_ground_validation_summary.csv")
    print(summary.round(4).to_string(index=False))
    C.ok("V06_ground_validation_summary.csv written")
else:
    C.warn("no pairs produced -- check that P08 linked at least one site inside this panel's footprint")
C.info("V06 | finished")
C.done(4, next_task="use V06_ground_validation_summary.csv in the paper; M07 runs on the GND_* outcomes built from the BM sub-watershed means (P00 P08b)")
